# 05 — Volatility Engine

ATR, realized volatility, EWMA volatility and volatility percentile.


In [2]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [3]:
# Cell: Import Volatility Modules
from src.data.loaders import read_parquet, save_parquet
from src.features.returns import add_returns
from src.features.volatility import (
    add_atr,
    add_realized_volatility,
    add_ewma_volatility,
    add_volatility_percentile,
    classify_volatility_regime,
)
from src.features.trend import add_moving_averages
from src.features.distribution import add_rolling_distribution_stats
from src.features.liquidity import add_liquidity_features


In [4]:
# Cell: Build Volatility Features
clean = read_parquet("data/processed/clean_daily_ohlcv.parquet")
features = add_returns(clean)
features = add_atr(features, window=14)
features = add_realized_volatility(features, window=20)
features = add_ewma_volatility(features, span=20)
features = add_volatility_percentile(features, vol_col="realized_vol_20d", window=252)
features = classify_volatility_regime(features)
features.shape


(188395, 24)

In [5]:
# Cell: Inspect Volatility Outputs
features[["symbol", "date", "close", "atr_pct_14", "realized_vol_20d", "volatility_percentile", "volatility_regime"]].sort_values(["symbol", "date"]).groupby("symbol").tail(3)


,symbol,date,close,atr_pct_14,realized_vol_20d,volatility_percentile,volatility_regime
65922,A32,2026-06-12,30000,0.044048,0.839738,NaN,Unknown
65923,A32,2026-06-17,30000,0.046667,0.811913,NaN,Unknown
65924,A32,2026-06-24,29300,0.049488,0.794251,NaN,Unknown
83275,AAA,2026-06-25,7430,0.020669,0.142061,0.051587,Low Vol
83276,AAA,2026-06-26,7370,0.020740,0.144172,0.067460,Low Vol
...,...,...,...,...,...,...,...
27025,YBM,2026-06-23,9990,NaN,0.291272,0.281746,Normal Vol
27026,YBM,2026-06-24,9990,NaN,0.269504,0.253968,Normal Vol
2563,YEG,2026-06-25,9550,0.023186,0.352505,0.416667,Normal Vol
2564,YEG,2026-06-26,9680,0.024424,0.339612,0.400794,Normal Vol


In [6]:
# Cell: Save Volatility Features & Full Stock Features
features = add_moving_averages(features, windows=[20, 50, 200])
features = add_rolling_distribution_stats(features, window=60)
features = add_liquidity_features(features)
save_parquet(features, "data/processed/stock_features.parquet")
features["volatility_regime"].value_counts(dropna=False)


volatility_regime
Unknown       135644
Low Vol        17448
Normal Vol     15154
High Vol       10479
Stress Vol      5524
Panic Vol       4146
Name: count, dtype: int64